# Araseの電磁場データについて、64 Hzデータを用いる。

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データのplot

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', no_update=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True)

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
E64_data_dsi_x  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform']
E64_data_dsi_y  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform']
B64_data_dsi    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E64_data_dsi_x  = E64_data_dsi_x.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = E64_data_dsi_y.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi    = B64_data_dsi.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

In [ ]:
import xarray as xr
import numpy as np

B64_data_dsi_time   = B64_data_dsi.time
E64_data_dsi_x      = E64_data_dsi_x.interp(time=B64_data_dsi_time, method='linear')
E64_data_dsi_y      = E64_data_dsi_y.interp(time=B64_data_dsi_time, method='linear')
vars_64             = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

E64_data_dsi_z      = xr.where(np.abs(B64_data_dsi.data[:, 2]) > 1E-12, -(E64_data_dsi_x.data * B64_data_dsi.data[:, 0] + E64_data_dsi_y.data * B64_data_dsi.data[:, 1]) / B64_data_dsi.data[:, 2], np.nan)

ds_64_dsi           = xr.Dataset({
    'E64_dsi_x':    E64_data_dsi_x,
    'E64_dsi_y':    E64_data_dsi_y,
    'E64_dsi_z':    ('time', E64_data_dsi_z),
    'B64_dsi_x':    ('time', B64_data_dsi.data[:, 0]),
    'B64_dsi_y':    ('time', B64_data_dsi.data[:, 1]),
    'B64_dsi_z':    ('time', B64_data_dsi.data[:, 2]),
}, coords={'time': B64_data_dsi_time})

ds_64_dsi   = ds_64_dsi.dropna(dim='time', how='all')

print(ds_64_dsi)

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

ds_64_dsi_segs  = split_by_gap(ds_64_dsi, gap_thr=np.timedelta64(8, 's'))

print(ds_64_dsi_segs)

ds_64_dsi_seg0  = ds_64_dsi_segs[0]

print(ds_64_dsi_seg0)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime

mpl.rcParams['font.size'] = 15
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_64_dsi_seg0_analysis  = ds_64_dsi_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_dsi_seg1_analysis  = ds_64_dsi_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_dsi_seg2_analysis  = ds_64_dsi_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_dsi_seg3_analysis  = ds_64_dsi_seg3.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime

mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['E64_dsi_x'], lw=1, c='blue')
#ax_1.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['E64_dsi_y'], lw=1, c='blue')
#ax_2.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['E64_dsi_z'], lw=1, c='blue')
#ax_3.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['B64_dsi_x'], lw=1, c='blue')
#ax_4.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['B64_dsi_y'], lw=1, c='blue')
#ax_5.plot(ds_64_dsi_seg0_analysis.time, ds_64_dsi_seg0_analysis['B64_dsi_z'], lw=1, c='blue')
#
##ax_0.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['E64_dsi_x'], lw=1, c='orange')
##ax_1.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['E64_dsi_y'], lw=1, c='orange')
##ax_2.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['E64_dsi_z'], lw=1, c='orange')
##ax_3.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['B64_dsi_x'], lw=1, c='orange')
##ax_4.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['B64_dsi_y'], lw=1, c='orange')
##ax_5.plot(ds_64_dsi_seg1_analysis.time, ds_64_dsi_seg1_analysis['B64_dsi_z'], lw=1, c='orange')
##
##ax_0.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['E64_dsi_x'], lw=1, c='green')
##ax_1.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['E64_dsi_y'], lw=1, c='green')
##ax_2.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['E64_dsi_z'], lw=1, c='green')
##ax_3.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['B64_dsi_x'], lw=1, c='green')
##ax_4.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['B64_dsi_y'], lw=1, c='green')
##ax_5.plot(ds_64_dsi_seg2_analysis.time, ds_64_dsi_seg2_analysis['B64_dsi_z'], lw=1, c='green')
##
##ax_0.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['E64_dsi_x'], lw=1, c='red')
##ax_1.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['E64_dsi_y'], lw=1, c='red')
##ax_2.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['E64_dsi_z'], lw=1, c='red')
##ax_3.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['B64_dsi_x'], lw=1, c='red')
##ax_4.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['B64_dsi_y'], lw=1, c='red')
##ax_5.plot(ds_64_dsi_seg3_analysis.time, ds_64_dsi_seg3_analysis['B64_dsi_z'], lw=1, c='red')
#
#ax_0.set_ylabel(r'$E_{x}$ (DSI)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (DSI)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (DSI)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (DSI)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (DSI)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (DSI)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_dsi_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# DSI -> FACに変換

In [ ]:
psp.erg.mgf(trange=time_range, level='l2', datatype='8sec', coord='dsi', no_update=True)
psp.erg.orb(trange=time_range, level='l2', datatype='def', no_update=True)

print(pt.tplot_names())

P6sec_data_gse      = pt.data_quants['erg_orb_l2_pos_gse'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B8sec_data_dsi      = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

P6sec_data_gse      = pt.data_quants['erg_orb_l2_pos_gse']
B100sec_data_dsi    = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].interp(time=P6sec_data_gse.time, method='linear')
B100sec_data_dsi    = B100sec_data_dsi.rolling(time=int(100/8), center=True).mean('time')

pt.store_data('P6sec_gse', data={'x': P6sec_data_gse.time, 'y': P6sec_data_gse.data}, attr_dict=P6sec_data_gse.attrs)
pt.store_data('B100sec_dsi', data={'x': P6sec_data_gse.time, 'y': B100sec_data_dsi.data}, attr_dict=B100sec_data_dsi.attrs)

from pyspedas.projects.erg.satellite.erg.particle.erg_pgs_make_fac import erg_pgs_make_fac

print(pt.data_quants['P6sec_gse'])
print(pt.data_quants['B100sec_dsi'])

FAC_matrix_np = erg_pgs_make_fac(
    P6sec_data_gse.time,
    mag_tvar_in='B100sec_dsi',
    pos_tvar_in='P6sec_gse',
    fac_type='mphism'
)

FAC_matrix_np   = FAC_matrix_np.astype(float)
pt.store_data('B100sec_dsi_FAC_matrix', data={'x': P6sec_data_gse.time, 'y': FAC_matrix_np})

FAC_matrix      = pt.data_quants['B100sec_dsi_FAC_matrix'].dropna(dim='time', how='any')
print(FAC_matrix)

In [ ]:
import numpy as np
import xarray as xr

# ---- 入力: FAC_matrix (xarray.DataArray, shape=(time, 3, 3)) ----
fac = FAC_matrix.values

# ---- 検証 ----
# 1. 直交性 (R R^T ≈ I)
orth_err = np.empty(fac.shape[0])
for i in range(fac.shape[0]):
    R = fac[i]
    I = np.eye(3)
    diff = R @ R.T - I
    orth_err[i] = np.linalg.norm(diff)  # Frobeniusノルム

# 2. 行列式
detR = np.linalg.det(fac)

# ---- 結果表示 ----
print("=== FAC_matrix orthogonality check ===")
print(f"orth_err mean: {orth_err.mean():.3e}, max: {orth_err.max():.3e}")
print(f"det(R) mean: {detR.mean():.6f}, std: {detR.std():.3e}")
print(f"det(R) range: {detR.min():.6f} – {detR.max():.6f}")

# しきい値を超えるサンプル検出（例: ノルム誤差 > 1e-6）
bad_idx = np.where((orth_err > 1e-6) | (np.abs(detR) < 0.999) | (np.abs(detR) > 1.001))[0]
if len(bad_idx) == 0:
    print("All matrices are orthonormal within tolerance.")
else:
    print(f"{len(bad_idx)} matrices deviate from orthonormality.")
    print("Example indices:", bad_idx[:10])


In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 5))
#ax = fig.add_subplot(111)
#ax.plot(FAC_matrix.time, np.rad2deg(np.arccos(FAC_matrix.data[:, 2, 2])), c='blue', lw=1)
#ax.minorticks_on()
#ax.grid(which='both', alpha=0.5)
#ax.set_ylabel(r'∠($\mathbf{B}_{0}$, $\mathbf{e}_{z}$ (DSI))')
#
#ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#ax.set_ylim(111, 126)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr

def rotate_dsi_to_fac(ds, fac_mat_da, e_base='E64_dsi', b_base='B64_dsi',
                      out_suffix='_fac', interp_method='linear'):
    """
    ds: xarray.Dataset（例: E8_dsi_x,y,z と B8_dsi_x,y,z を持つ）
    fac_mat_da: (time, 3, 3) の回転行列 DataArray（DSI→FAC）
    e_base/b_base: 先頭名
    out_suffix: 出力成分名の接尾辞（_fac）
    """

    M = fac_mat_da.interp(time=ds.time, method=interp_method)

    def vec3(base):
        v = xr.concat([ds[f'{base}_x'], ds[f'{base}_y'], ds[f'{base}_z']], dim='c')
        return v.transpose('time', 'c').astype(np.float64)  # (time,3)

    def matvec(M, V):
        out = np.einsum('tij,tj->ti', M.values, V.values)
        return xr.DataArray(out, coords={'time': V['time'], 'c': ['x','y','z']},
                            dims=['time','c'])

    def split_drop(V):
        # c 座標を削除して 1D に
        x = V.sel(c='x').reset_coords('c', drop=True)
        y = V.sel(c='y').reset_coords('c', drop=True)
        z = V.sel(c='z').reset_coords('c', drop=True)
        return x, y, z

    E_fac = matvec(M, vec3(e_base))
    B_fac = matvec(M, vec3(b_base))

    Ex, Ey, Ez = split_drop(E_fac)
    Bx, By, Bz = split_drop(B_fac)

    def _stem(name: str) -> str:
        # 末尾のアンダースコア区切りを1つだけ落とす（'E8_dsi'→'E8'）
        return name.rsplit('_', 1)[0] if '_' in name else name
    
    e_stem = _stem(e_base)
    b_stem = _stem(b_base)
    
    return xr.Dataset(
          {f'{e_stem}{out_suffix}_x': Ex,
           f'{e_stem}{out_suffix}_y': Ey,
           f'{e_stem}{out_suffix}_z': Ez,
           f'{b_stem}{out_suffix}_x': Bx,
           f'{b_stem}{out_suffix}_y': By,
           f'{b_stem}{out_suffix}_z': Bz},
           attrs=ds.attrs
    )

In [ ]:
ds_64_fac_seg0   = rotate_dsi_to_fac(ds=ds_64_dsi_seg0, fac_mat_da=FAC_matrix, e_base='E64_dsi', b_base='B64_dsi')
#ds_64_fac_seg1   = rotate_dsi_to_fac(ds=ds_64_dsi_seg1, fac_mat_da=FAC_matrix, e_base='E64_dsi', b_base='B64_dsi')
#ds_64_fac_seg2   = rotate_dsi_to_fac(ds=ds_64_dsi_seg2, fac_mat_da=FAC_matrix, e_base='E64_dsi', b_base='B64_dsi')
#ds_64_fac_seg3   = rotate_dsi_to_fac(ds=ds_64_dsi_seg3, fac_mat_da=FAC_matrix, e_base='E64_dsi', b_base='B64_dsi')

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_64_fac_seg0_analysis  = ds_64_fac_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_fac_seg1_analysis  = ds_64_fac_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_fac_seg2_analysis  = ds_64_fac_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
##ds_64_fac_seg3_analysis  = ds_64_fac_seg3.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['E64_fac_x'], lw=1, c='blue')
#ax_1.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['E64_fac_y'], lw=1, c='blue')
#ax_2.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['E64_fac_z'], lw=1, c='blue')
#ax_3.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['B64_fac_x'], lw=1, c='blue')
#ax_4.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['B64_fac_y'], lw=1, c='blue')
#ax_5.plot(ds_64_fac_seg0_analysis.time, ds_64_fac_seg0_analysis['B64_fac_z'], lw=1, c='blue')
#
##ax_0.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['E64_fac_x'], lw=1, c='orange')
##ax_1.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['E64_fac_y'], lw=1, c='orange')
##ax_2.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['E64_fac_z'], lw=1, c='orange')
##ax_3.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['B64_fac_x'], lw=1, c='orange')
##ax_4.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['B64_fac_y'], lw=1, c='orange')
##ax_5.plot(ds_64_fac_seg1_analysis.time, ds_64_fac_seg1_analysis['B64_fac_z'], lw=1, c='orange')
##
##ax_0.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['E64_fac_x'], lw=1, c='green')
##ax_1.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['E64_fac_y'], lw=1, c='green')
##ax_2.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['E64_fac_z'], lw=1, c='green')
##ax_3.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['B64_fac_x'], lw=1, c='green')
##ax_4.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['B64_fac_y'], lw=1, c='green')
##ax_5.plot(ds_64_fac_seg2_analysis.time, ds_64_fac_seg2_analysis['B64_fac_z'], lw=1, c='green')
##
##ax_0.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['E64_fac_x'], lw=1, c='red')
##ax_1.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['E64_fac_y'], lw=1, c='red')
##ax_2.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['E64_fac_z'], lw=1, c='red')
##ax_3.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['B64_fac_x'], lw=1, c='red')
##ax_4.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['B64_fac_y'], lw=1, c='red')
##ax_5.plot(ds_64_fac_seg3_analysis.time, ds_64_fac_seg3_analysis['B64_fac_z'], lw=1, c='red')
#
#ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_fac_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
t = ds_64_fac_seg0.time.values.astype('datetime64[ns]').astype('int64')/1e9
true_dt = np.median(np.diff(t))
print("true_dt =", true_dt)      # 例: 0.0078125 (= 1/128)
print("your_dt =", 1/64)         # 渡している dt
print("ratio  =", (1/64)/true_dt)

# Wavelet analysis

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

sys.path.append("..")
import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

ds_64_fac_seg0 = ds_64_fac_seg0.interpolate_na(dim='time', method='linear')

vars_64 = ['E64_fac_x','E64_fac_y','E64_fac_z', 'B64_fac_x','B64_fac_y','B64_fac_z']
ds_64_fac_cwt_seg0 = tw.cwt_from_dataset(ds_64_fac_seg0, dt=1/64, s0=2, dj=1/32, variables=vars_64)
#ds_64_fac_cwt_seg1 = tw.cwt_from_dataset(ds_64_fac_seg1, dt=1/64, s0=1, dj=1/32, variables=vars_64)
#ds_64_fac_cwt_seg2 = tw.cwt_from_dataset(ds_64_fac_seg2, dt=1/64, s0=1, dj=1/32, variables=vars_64)
#ds_64_fac_cwt_seg3 = tw.cwt_from_dataset(ds_64_fac_seg3, dt=1/64, s0=1, dj=1/32, variables=vars_64)

print(ds_64_fac_cwt_seg0)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
#dsets_64 = [ds_64_fac_cwt_seg0]
#targets = [
#    ("E64_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B64_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#joined = {}
#for v, _, _ in targets:
#    da, coi = concat_cwt_segments(dsets_64, v)
#    da = da.sortby('freq')
#    if da is not None: joined[v] = (da.sortby("freq"), coi)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined: continue
#        da, coi = joined[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 各成分のNoise(Median)を抽出

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# 入力: ds_8_fac_cwt_seg0
noise_t0, noise_t1 = np.datetime64('2022-09-01T21:30:00'), np.datetime64('2022-09-01T22:00:00')

# 対象変数（存在チェック付き）
vars_64 = [
    'E64_fac_x_cwt','E64_fac_y_cwt','E64_fac_z_cwt',
    'B64_fac_x_cwt','B64_fac_y_cwt','B64_fac_z_cwt'
]

# 時間で切り出し → 周波数ごとに時間方向のnanmedian
noise_da_dict = {}
freq_ref = None
for v in vars_64:
    da = ds_64_fac_cwt_seg0[v].sel(time=slice(noise_t0, noise_t1))
    if da.time.size == 0:
        continue
    med = np.nanmedian(da.values, axis=0)  # (freq,)
    freq = da.coords['freq'].values
    if freq_ref is None:
        freq_ref = freq
    noise_da_dict[v] = xr.DataArray(med, dims=['frequency'], coords={'frequency': freq_ref}, name=v)

noise_ds = xr.Dataset(noise_da_dict)
print(noise_ds)

In [ ]:
# プロット
#fig, ax = plt.subplots(figsize=(8,6))
#for v in noise_ds.data_vars:
#    # ラベル例: E_x, B_y など
#    prefix = v.split('_')[0]   # E8 or B8
#    comp   = v.split('_')[2]   # x/y/z
#    label  = f"${prefix[0]}_{comp}$"
#    ax.loglog(noise_ds['frequency'], noise_ds[v], label=label)
#
#ax.minorticks_on()
#ax.set_xlabel('Frequency [Hz]')
#ax.set_ylabel('Median PSD')
#ax.set_title('Noise floor (median) 21:30–22:00  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)')
#ax.grid(True, which='both', ls=':')
#ax.legend(ncol=3)
#ax.set_xlim(1e-2, 32)
#ax.set_ylim(1e-8, 5e1)
#plt.tight_layout()
#
## 保存 or 表示
#if os.path.isdir(path_base_save_plot):
#    fn = f"noise_median_bs_{str(noise_t0).replace(':','')}_{str(noise_t1).replace(':','')}.png"
#    fig.savefig(os.path.join(path_base_save_plot, fn), dpi=300, bbox_inches='tight')
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import xarray as xr
import numpy as np

def make_cwt_clean_segments_64(dsets_64, noise_ds):
    """
    入力:
      dsets_64 : [ds_64_fac_cwt_seg0]
      noise_ds: 各成分のノイズ床（frequency or freq 次元, 1D）
    出力:
      cleaned_64: [ds_64_fac_cwt_clean_seg0, ...]  各dsは
                 {E64_fac_{x,y,z}_cwt_clean, B64_fac_{x,y,z}_cwt_clean,
                  E64_fac_{x,y,z}_coi,      B64_fac_{x,y,z}_coi} をdata_varsに持つ
    """
    # 周波数座標名を統一
    def _noise_for(var):
        if var not in noise_ds:
            return None
        nda = noise_ds[var]
        if "frequency" in nda.dims:
            nda = nda.rename({"frequency": "freq"})
        return nda

    target_vars = [
        "E64_fac_x_cwt","E64_fac_y_cwt","E64_fac_z_cwt",
        "B64_fac_x_cwt","B64_fac_y_cwt","B64_fac_z_cwt",
    ]

    cleaned_list = []
    for ds in dsets_64:
        new_vars = {}
        # 座標をそのまま流用
        coords = {"time": ds.time, "freq": ds.freq}

        for v in target_vars:
            if v not in ds:
                continue
            n_da = _noise_for(v)
            if n_da is None:
                continue

            # ノイズ床を各dsのfreqに合わせる
            n_interp = n_da.interp(freq=ds[v].freq)

            # 減算（broadcast）
            cleaned = ds[v] - n_interp
            cleaned = cleaned.where(cleaned > 0)

            new_vars[f"{v}_clean"] = xr.DataArray(
                cleaned.astype(np.float32),
                dims=("time", "freq"),
                coords=coords,
                attrs={**ds[v].attrs, "noise_removed": True}
            )

            # 対応するCOIをそのまま持たせる
            coi_name = v.replace("_cwt", "_coi")
            if coi_name in ds:
                new_vars[coi_name] = ds[coi_name]

        cleaned_list.append(xr.Dataset(new_vars, coords=coords))

    return cleaned_list

# 使い方
dsets_64 = [ds_64_fac_cwt_seg0]
ds_64_fac_cwt_clean_segs = make_cwt_clean_segments_64(dsets_64, noise_ds)

ds_64_fac_cwt_clean_seg0 = ds_64_fac_cwt_clean_segs[0]

print(ds_64_fac_cwt_clean_seg0)

In [ ]:
#dsets_64 = ds_64_fac_cwt_clean_seg0
#targets = [
#    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#joined = {}
#for v, _, _ in targets:
#    da, coi = concat_cwt_segments(dsets_64, v)
#    da = da.sortby('freq')
#    if da is not None: joined[v] = (da.sortby("freq"), coi)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined: continue
#        da, coi = joined[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
mu_0 = 4.*np.pi*1E-7

S_para  = (ds_64_fac_seg0['E64_fac_x'] * ds_64_fac_seg0['B64_fac_y'] - ds_64_fac_seg0['E64_fac_y'] * ds_64_fac_seg0['B64_fac_x']) /mu_0 * 1E-12

print(S_para[370000:370100]*1E3)

In [ ]:
dsets_64 = ds_64_fac_cwt_clean_seg0
targets = [
    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T22:35:00'),
    np.datetime64('2022-09-01T22:50:00'),
    np.datetime64('2022-09-01T23:07:30')
]

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets)+1, 1, figsize=(10, 12), sharex=True)
    axes_cwt = axes[:len(targets)]
    for ax, (v, ylab, unit) in zip(axes_cwt, targets):
        if v not in joined: continue
        da, coi = joined[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    
    S_para_window   = S_para.sel(time=slice(t0, t0+np.timedelta64(5, 'm')))
    axes[len(targets)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    axes[len(targets)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    axes[len(targets)].minorticks_on()
    axes[len(targets)].grid(which='both', alpha=0.5)
    axes[-1].set_xlabel("time")
    fig.tight_layout()
    fig.subplots_adjust(hspace=0.1)

    add_panel_label(axes[0], '(1)')
    add_panel_label(axes[1], '(2)')
    add_panel_label(axes[2], '(3)')
    add_panel_label(axes[3], '(4)')
    add_panel_label(axes[4], '(5)')
    add_panel_label(axes[5], '(6)')
    add_panel_label(axes[6], '(7)')

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}_for_figure.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# 軌道データから、衛星速度(DSI)を導出

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

psp.erg.orb(trange=time_range, level='l2', datatype='def', no_update=True)

pos   = pt.data_quants['erg_orb_l2_pos_gse'].data   # (Nt,3)[R_E]
t_pos = pt.data_quants['erg_orb_l2_pos_gse'].time.values

R_E_m = 6.378137e6
t_s   = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
v_gse = np.gradient(pos * R_E_m, t_s, axis=0)

v_sc_gse = xr.DataArray(
    data=v_gse, dims=('time','v_dim'),
    coords={'time': t_pos},
    attrs={'units':'m/s','desc':'$V_{sc}$ (GSE)'}
)
v_sc_gse.name = 'v_sc_gse'

print(v_sc_gse)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_gse_analysis  = v_sc_gse.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSE)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSE)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSE)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gse.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import numpy as np, pytplot as pt, pyspedas as psp
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

def store64(name, t, y):
    pt.store_data(name, data={'x': t.astype('datetime64[ns]'),
                              'y': np.asarray(y, dtype=np.float64)})

# 64bitで登録
store64('v_sc_gse_64', v_sc_gse.time.values, v_sc_gse.data)

# 変換
psp.cotrans(name_in='v_sc_gse_64', name_out='v_sc_j2000_64', coord_in='gse', coord_out='j2000')
dsi2j2000(name_in='v_sc_j2000_64', name_out='v_sc_dsi_64', J20002DSI=True)

# 検証
def vnorm(name): dq=pt.data_quants[name]; return np.linalg.norm(dq.data,axis=1)
ng, nj, nd = vnorm('v_sc_gse_64'), vnorm('v_sc_j2000_64'), vnorm('v_sc_dsi_64')

def check(a,b,tag,thr_rel=1e-12):
    rel = np.abs(a-b)/np.maximum(a,1e-30)
    print(f"{tag}: rel mean={rel.mean():.3e}, max={rel.max():.3e}")
    assert np.nanmax(rel) < thr_rel, f"{tag} norm not preserved"

check(ng,nj,"GSE→J2000")
check(nj,nd,"J2000→DSI")

v_sc_j2000  = pt.data_quants['v_sc_j2000_64']
v_sc_dsi    = pt.data_quants['v_sc_dsi_64']

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_j2000_analysis  = v_sc_j2000.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (J2000)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (J2000)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (J2000)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_j2000.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_dsi_analysis  = v_sc_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr

def rotate_vec3_dsi_to_fac(da_vec, fac_mat_da, out_suffix="_fac", interp_method="linear"):
    """
    DSI→FAC回転を v_dim=3 のベクトル DataArray に適用する。

    Parameters
    ----------
    da_vec : xarray.DataArray
        形状 (time, v_dim=3) のベクトルデータ。座標 'time' と 'v_dim' を含む。
    fac_mat_da : xarray.DataArray
        形状 (time, 3, 3) の回転行列（DSI→FAC変換行列）。
    out_suffix : str
        出力名の接尾辞。
    interp_method : str
        時間補間法。'linear' など。

    Returns
    -------
    da_fac : xarray.DataArray
        形状 (time, v_dim=3) の回転後ベクトル。
        名前は da_vec.name + out_suffix。
    """
    # 時間軸を補間
    M = fac_mat_da.interp(time=da_vec.time, method=interp_method)

    # einsum でベクトル回転
    V_in = da_vec.transpose("time", "v_dim").astype(np.float64)
    V_out = np.einsum("tij,tj->ti", M.values, V_in.values)  # (time, 3)

    # 結果をDataArrayに再構成
    da_fac = xr.DataArray(
        V_out.astype(np.float32),
        dims=("time", "v_dim"),
        coords={"time": da_vec.time, "v_dim": da_vec.v_dim},
        name=(f"{da_vec.name}{out_suffix}" if da_vec.name else None),
        attrs={**da_vec.attrs, "rotated": "DSI→FAC"},
    )
    return da_fac

In [ ]:
v_sc_fac    = rotate_vec3_dsi_to_fac(v_sc_dsi, FAC_matrix)
v_sc_fac    = v_sc_fac.dropna(dim='time', how='any')
print(v_sc_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.erg.lepe(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.lepi(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.pwe_hfa(trange=time_range, level='l3', no_update=True)

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FHEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FODU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

In [ ]:
ND_electron_LEP = pt.data_quants['erg_lepe_l2_3dflux_FEDU_density']
ND_electron_HFA = pt.data_quants['erg_pwe_hfa_l3_1min_ne_mgf']
Temp_electron   = pt.data_quants['erg_lepe_l2_3dflux_FEDU_avgtemp']

ND_proton       = pt.data_quants['erg_lepi_l2_3dflux_FPDU_density'].fillna(0)   # [/cc]
Flux_proton     = pt.data_quants['erg_lepi_l2_3dflux_FPDU_flux'].fillna(0)      # [/s/cm2]
Temp_proton     = pt.data_quants['erg_lepi_l2_3dflux_FPDU_avgtemp'].fillna(0)   # [eV]

ND_Helium       = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_density'].fillna(0)
Flux_Helium     = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_flux'].fillna(0)
Temp_Helium     = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_avgtemp'].fillna(0)

ND_Oxygen       = pt.data_quants['erg_lepi_l2_3dflux_FODU_density'].fillna(0)
Flux_Oxygen     = pt.data_quants['erg_lepi_l2_3dflux_FODU_flux'].fillna(0)
Temp_Oxygen     = pt.data_quants['erg_lepi_l2_3dflux_FODU_avgtemp'].fillna(0)

ND_ion          = ND_proton + ND_Helium + ND_Oxygen                                     # [/cc]
Flux_ion        = Flux_proton + Flux_Helium + Flux_Oxygen                               # [/s/cm2]
Ptot_ion        = ND_proton*Temp_proton + ND_Helium*Temp_Helium + ND_Oxygen*Temp_Oxygen # [eV/cc]

v_ion_dsi       = Flux_ion / ND_ion * 1E-2  # [m/s]
Temp_ion        = Ptot_ion / ND_ion         # [eV]

proton_mass_kg  = 1.6726219e-27  # kg
Helium_mass_kg  = proton_mass_kg * 4
Oxygen_mass_kg  = proton_mass_kg * 16

ion_mass        = (ND_proton*proton_mass_kg + ND_Helium*Helium_mass_kg + ND_Oxygen*Oxygen_mass_kg) / ND_ion #[kg]

In [ ]:
v_ion_fac   = rotate_vec3_dsi_to_fac(v_ion_dsi, FAC_matrix)
v_ion_fac   = v_ion_fac.dropna(dim='time', how='any')
print(v_ion_fac.time)
print(v_sc_fac.time)

In [ ]:
v_sys_fac   = v_ion_fac.interp(time=v_sc_fac.time, method='linear') - v_sc_fac
v_sys_fac   = v_sys_fac.dropna(dim='time', how='any')
print(v_sys_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_dsi_analysis  = v_ion_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
#v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ND_proton_analysis  = ND_proton.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis  = ND_Helium.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis  = ND_Oxygen.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis     = ND_ion.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis  = ion_mass.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis.time,  ND_proton_analysis.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis.time,  ND_Helium_analysis.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis.time,  ND_Oxygen_analysis.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis.time,     ND_ion_analysis.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis.time,   ion_mass_analysis.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_ND               = (ND_proton.time.data[1] - ND_proton.time.data[0]) / np.timedelta64(1, 's')
#
#ND_proton_mean  = ND_proton.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Helium_mean  = ND_Helium.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Oxygen_mean  = ND_Oxygen.rolling(time=int(100/dt_ND), center=True).mean()
#ND_ion_mean     = ND_ion.rolling(time=int(100/dt_ND), center=True).mean()
#ion_mass_mean   = ion_mass.rolling(time=int(100/dt_ND), center=True).mean()
#
#ND_proton_analysis_mean  = ND_proton_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis_mean  = ND_Helium_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis_mean  = ND_Oxygen_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis_mean     = ND_ion_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis_mean   = ion_mass_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis_mean.time,  ND_proton_analysis_mean.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis_mean.time,  ND_Helium_analysis_mean.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis_mean.time,  ND_Oxygen_analysis_mean.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis_mean.time,     ND_ion_analysis_mean.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis_mean.time,   ion_mass_analysis_mean.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [ ]:
B_total = pt.data_quants['erg_mgf_l2_magt_8sec']    # [nT]

time_base = B_total.time
print(time_base)

ND_electron_LEP_interp  = ND_electron_LEP.interp(time=time_base, method='linear')
ND_electron_HFA_interp  = ND_electron_HFA.interp(time=time_base, method='linear')
ND_electron_MID_interp  = (ND_electron_HFA_interp + ND_electron_LEP_interp) / 2E0


Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
v_sys_fac_interp        = v_sys_fac.interp(time=time_base, method='linear')
ion_mass_interp         = ion_mass.interp(time=time_base, method='linear')

v_sys_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_sys_fac_interp.data[:, 0]**2E0 + v_sys_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_sys_fac_interp.time},
    attrs=v_sys_fac_interp.attrs
)

elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed_LEP    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_LEP_interp*1E6 * ion_mass_interp)
Alfven_speed_HFA    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_HFA_interp*1E6 * ion_mass_interp)
Alfven_speed_MID    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_MID_interp*1E6 * ion_mass_interp)

ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / ion_mass_interp)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / ion_mass_interp)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass_kg / 2E0 / np.pi

ion_plasma_beta_LEP = (ion_thermal_speed / Alfven_speed_LEP)**2E0
ion_plasma_beta_HFA = (ion_thermal_speed / Alfven_speed_HFA)**2E0
ion_plasma_beta_MID = (ion_thermal_speed / Alfven_speed_MID)**2E0

ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[1] - time_base.data[0]) / np.timedelta64(1, 's')

Alfven_speed_LEP_mean   = Alfven_speed_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_HFA_mean   = Alfven_speed_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_MID_mean   = Alfven_speed_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_perp_mean     = v_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_LEP_mean            = ion_plasma_beta_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_HFA_mean            = ion_plasma_beta_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_MID_mean            = ion_plasma_beta_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()

# DataSet格納
ds_velocity_ms = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           v_sys_fac_perp_mean
    }
)
ds_velocity_ms = ds_velocity_ms.dropna(dim='time', how='any')

print(ds_velocity_ms)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta_LEP':      ion_plasma_beta_LEP_mean,
        'ion_plasma_beta_MID':      ion_plasma_beta_MID_mean,
        'ion_plasma_beta_HFA':      ion_plasma_beta_HFA_mean,
        'i-e_temp_ratio':           ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':      proton_cycl_freq_mean,
        'number_density_LEP_cc':    ND_electron_LEP_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_MID_cc':    ND_electron_MID_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_HFA_cc':    ND_electron_HFA_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'ion_mass_kg':              ion_mass_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':              Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':         Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':               B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter.dropna(dim='time', how='any')

print(ds_parameter)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       lw=1, c='b')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       lw=1, c='k')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       lw=1, c='r')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 15))
#gs = fig.add_gridspec(8, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_LEP_cc'],       lw=1, c='b')
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],       lw=1, c='k')
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_HFA_cc'],       lw=1, c='r')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_mass_kg']/proton_mass_kg,  lw=1, c='k')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['i-e_temp_ratio'],              lw=1, c='k')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_LEP'],         lw=1, c='b')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],         lw=1, c='k')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_HFA'],         lw=1, c='r')
#ax_5.plot(ds_parameter_analysis.time, electron_mass_kg/ds_parameter_analysis['ion_mass_kg'], lw=2, c='green', linestyle='-.')
#ax_6.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k')
#ax_7.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'],         lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$m_{\mathrm{i}}$'     + '\n' + r'[$m_{\mathrm{p}}$]')
#ax_2.set_ylabel(r'$T_{\mathrm{i}}$'     + '\n' + '[eV]')
#ax_3.set_ylabel(r'$T_{\mathrm{e}}$'     + '\n' + '[eV]')
#ax_4.set_ylabel(r'$\tau$')
#ax_5.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_6.set_ylabel(r'$B_{0}$'              + '\n' + '[nT]')
#ax_7.set_ylabel(r'$f_{\mathrm{H}^{+}}$' + '\n' + '[Hz]')
#
#ax_0.set_yscale('log')
#ax_5.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'parameter_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_analysis = ds_velocity_ms.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#
#mpl.rcParams['font.size'] = 25
#
#fig = plt.figure(figsize=(11, 21))
#gs = fig.add_gridspec(8, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_2_share = ax_2.twinx()
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_LEP_cc'],       lw=1, c='green', label='LEP')
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],       lw=1, c='k', label='MID')
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_HFA_cc'],       lw=1, c='orange', label='HFA')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'],         lw=1, c='k')
#ax_2_share.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],            lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_mass_kg']/proton_mass_kg,  lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_LEP'],         lw=1, c='green', label='LEP')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],         lw=1, c='k', label='MID')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_HFA'],         lw=1, c='orange', label='HFA')
#ax_4.plot(ds_parameter_analysis.time, electron_mass_kg/ds_parameter_analysis['ion_mass_kg'],lw=2, c='purple', linestyle='-.', label=r'$m_{\mathrm{e}}/m_{\mathrm{i}}$')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       lw=1, c='green', label=r'$v_{\mathrm{A}}$ (LEP)')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       lw=1, c='k', label=r'$v_{\mathrm{A}}$ (MID)')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       lw=1, c='orange', label=r'$v_{\mathrm{A}}$ (HFA)')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=2, c='b', linestyle='-.', label=r'$v_{\mathrm{the}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      lw=2, c='red', linestyle='-.', label=r'$v_{\mathrm{thi}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     lw=1, c='magenta', label=r'$c_{\mathrm{s}}$')
#ax_7.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
#ax_2.set_ylabel(r'$f_{\mathrm{H}^{+}}$'                 + '\n' + '[Hz]')
#ax_2_share.set_ylabel(r'$B_{0}$'                        + '\n' + '[nT]')
#ax_3.set_ylabel(r'$m_{\mathrm{i}}$'                     + '\n' + r'[$m_{\mathrm{H}}$]')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$v_{\mathrm{A}}$, $v_{\mathrm{the}}$' + '\n' + '[km/s]')
#ax_6.set_ylabel(r'$v_{\mathrm{thi}}$, $c_{\mathrm{s}}$' + '\n' + '[km/s]')
#ax_7.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.set_yscale('log')
#ax_1.set_yscale('log')
#ax_3.set_ylim(ymin=1)
#ax_4.set_yscale('log')
#ax_5.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)
#
#ax_0.legend(fontsize=13, ncol=3)
#ax_1.legend(fontsize=13, ncol=2)
#ax_4.legend(fontsize=13, ncol=4)
#ax_5.legend(fontsize=13, ncol=4)
#ax_6.legend(fontsize=13, ncol=2)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#def add_panel_label(ax, label, x=-0.15, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#pos_da = pt.data_quants['erg_orb_l2_pos_rmlatmlt']  # (Nt, 3)
#t_pos_py = to_py_datetime(pos_da.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(pos_da.values[:, 0], dtype=float)  # Re
#mlat= np.asarray(pos_da.values[:, 1], dtype=float)  # deg
#mlt = np.asarray(pos_da.values[:, 2], dtype=float)  # hour [0,24)
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=60)  # R/MLAT/MLTラベル
#
## 好みで：目盛間隔をメイン x と合わせる
#secax.set_ticks(ax_7.get_xticks())
#
#fig.text(0.05, 0.085, "hhmm", ha='center', va='center')
#fig.text(0.05, 0.048, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
#fig.text(0.05, 0.028, r"MLAT", ha='center', va='center')
#fig.text(0.05, 0.008, r"MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(a-1)')
#add_panel_label(ax_1, '(a-2)')
#add_panel_label(ax_2, '(a-3)')
#add_panel_label(ax_3, '(a-4)')
#add_panel_label(ax_4, '(a-5)')
#add_panel_label(ax_5, '(a-6)')
#add_panel_label(ax_6, '(a-7)')
#add_panel_label(ax_7, '(a-8)')
#
#fig.suptitle('Arase', y=0.99)
#
#fig.subplots_adjust(hspace=0)
#fig.tight_layout(pad=0)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_2a.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter_clean = dedup_and_sort(ds_parameter)
ds_velocity_ms_clean = dedup_and_sort(ds_velocity_ms)

ds_parameter_interp = ds_parameter_clean.interp(time=ds_velocity_ms_clean.time)
print(ds_parameter_interp)
print(ds_velocity_ms_clean)

In [ ]:
T_window = ds_parameter_interp['ion_mass_kg'] / proton_mass_kg / ds_parameter_interp['proton_cycl_freq_Hz'].data * ds_velocity_ms_clean['ion_thermal_speed'].data / ds_velocity_ms_clean['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter_interp['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter_interp['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter_interp['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_clean.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

mpl.rcParams['font.size'] = 25
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
ax.minorticks_on()
ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
ax.set_yscale('log')
ax.set_ylim(ymin=0.1)
ax.grid(which='both', alpha=0.5)
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

fig.tight_layout()
plt.show(fig)
print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))

In [ ]:
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

dsets_64    = ds_64_fac_cwt_clean_seg0

dd  = psdpAx.build_data_dict_xr_erg(dsets_64, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 1/8*0.7, 1/8*5, 32])

fig1 = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64('2022-09-01T22:37:00'), dt_sec=0.5, variant="mid")
plt.show(fig1)
fig2, _ = psdpAx.plot_k_spectrum_erg(dd,   np.datetime64('2022-09-01T22:37:00'), dt_sec=0.5, n_bins=30)
plt.show(fig2)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    ds_64_fac_cwt_clean_seg0, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    ds_64_fac_cwt_clean_seg0, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    ds_64_fac_cwt_clean_seg0, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:37:30', '2022-09-01T22:38:15']
#time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_Arase_xarray as psdpAx
importlib.reload(psdpAx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.5
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    ds_64_fac_cwt_clean_seg0, ds_velocity_ms, ds_parameter, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")